<a href="https://colab.research.google.com/github/irumsultana/VaR-CALCULATOR-/blob/main/VaR_CALCULATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
-#VALUATION AT RISK CALCULATOR

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

# 1. Fetch Data
# Using 'AAPL' as an example; for PSX use 'LUCK.KA' or '^KSE100'
ticker = "AAPL"
data = yf.download(ticker, start="2023-01-01", end="2026-04-25")

# 2. Calculate Daily Returns
# We use log returns for financial modeling as they are additive
returns = np.log(data['Close'] / data['Close'].shift(1)).dropna()

def calculate_vars(returns, confidence_level=0.95):
    """
    Calculates VaR using Parametric, Historical, and Monte Carlo methods.
    Returns a dictionary of results.
    """
    alpha = 1 - confidence_level

    # --- PARAMETRIC VaR ---
    mu = np.mean(returns)
    sigma = np.std(returns)
    # norm.ppf finds the percentile on the normal distribution curve
    p_var = mu + norm.ppf(alpha) * sigma

    # --- HISTORICAL VaR ---
    # Simply finding the actual 5th percentile of past data
    h_var = np.percentile(returns, alpha * 100)

    # --- MONTE CARLO VaR ---
    np.random.seed(42) # For reproducible results
    sim_iterations = 10000
    sim_returns = np.random.normal(mu, sigma, sim_iterations)
    m_var = np.percentile(sim_returns, alpha * 100)

    return {
        "Parametric VaR": abs(p_var),
        "Historical VaR": abs(h_var),
        "Monte Carlo VaR": abs(m_var)
    }

# 3. Execute and Print Results
results = calculate_vars(returns)

print(f"--- VaR Results for {ticker} (95% Confidence) ---")
for method, value in results.items():
    # Check if the value is a pandas Series and extract the scalar
    if isinstance(value, pd.Series):
        value_to_format = value.item() # Extracts the scalar from a single-element Series
    else:
        value_to_format = value
    print(f"{method}: {value_to_format:.2%}")

# Optional: Portfolio Impact
portfolio_value = 1000000 # e.g., 1 Million PKR/USD
# Ensure all VaR values are scalars before multiplication and formatting
parametric_var_scalar = results['Parametric VaR'].item() if isinstance(results['Parametric VaR'], pd.Series) else results['Parametric VaR']
historical_var_scalar = results['Historical VaR'].item() if isinstance(results['Historical VaR'], pd.Series) else results['Historical VaR']
monte_carlo_var_scalar = results['Monte Carlo VaR'].item() if isinstance(results['Monte Carlo VaR'], pd.Series) else results['Monte Carlo VaR']

print(f"\nPotential 1-Day Loss (Parametric): {parametric_var_scalar * portfolio_value:,.2f}")
print(f"Potential 1-Day Loss (Historical): {historical_var_scalar * portfolio_value:,.2f}")
print(f"Potential 1-Day Loss (Monte Carlo): {monte_carlo_var_scalar * portfolio_value:,.2f}")

/tmp/ipykernel_2170/2162617513.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start="2023-01-01", end="2026-04-25")
[*********************100%***********************]  1 of 1 completed

--- VaR Results for AAPL (95% Confidence) ---
Parametric VaR: 2.54%
Historical VaR: 2.43%
Monte Carlo VaR: 2.55%

Potential 1-Day Loss (Parametric): 25,353.76
Potential 1-Day Loss (Historical): 24,343.70
Potential 1-Day Loss (Monte Carlo): 25,513.76



/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3800: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
